# Credit Card Fraud Detection: Handling Severe Class Imbalance

**Objective:** build a machine-learning pipeline that detects fraudulent transactions in a heavily imbalanced dataset, treating class imbalance as the core challenge.

**Dataset:** [Kaggle: Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) (284,807 transactions, 492 fraudulent). Features `V1`-`V28` are PCA components (anonymised), plus `Time` (seconds since first transaction), `Amount`, and `Class` (1 = fraud).

**Pipeline**

1. Load data and quantify imbalance
2. EDA (amount distribution, time-of-day)
3. Why accuracy is misleading
4. Stratified split, then imbalance handling (SMOTE and `class_weight`)
5. Models: Logistic Regression and Random Forest
6. Evaluation: Precision, Recall, F1, ROC-AUC, PR-AUC
7. Recall vs. precision, and threshold tuning
8. Feature importance / coefficients
9. Scalability: 1 million transactions per hour

> **How to run:** `Kernel -> Restart & Run All`. The notebook uses `creditcard.csv` if it sits next to it (or in `./data/`), otherwise tries to download the real dataset via `kagglehub`. Only if both fail does it generate a *synthetic* dataset with the same schema so every cell still runs; the numbers will then differ from the real benchmark.

In [ ]:
# pip install pandas numpy scikit-learn imbalanced-learn matplotlib seaborn
import time, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix, classification_report)

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load the data and analyse class imbalance

In [ ]:
def make_synthetic_fraud_data(n=100_000, fraud_rate=0.004, seed=RANDOM_STATE):
    """Fallback only: mimics the schema of the Kaggle dataset (Time, V1..V28, Amount, Class)."""
    from sklearn.datasets import make_classification
    rng = np.random.default_rng(seed)
    X, y = make_classification(n_samples=n, n_features=28, n_informative=12, n_redundant=4,
                               n_clusters_per_class=2, weights=[1 - fraud_rate, fraud_rate],
                               class_sep=1.4, flip_y=0.001, random_state=seed)
    df = pd.DataFrame(X, columns=[f"V{i}" for i in range(1, 29)])
    df["Class"] = y
    # Legit traffic follows a day/night cycle; fraud is flatter and skews to night hours
    hours = np.where(y == 0,
                     rng.choice(24, n, p=np.array([1,1,1,1,1,2,4,6,8,9,9,9,9,9,9,9,9,9,9,8,7,5,3,2], float) /
                                np.array([1,1,1,1,1,2,4,6,8,9,9,9,9,9,9,9,9,9,9,8,7,5,3,2], float).sum()),
                     rng.choice(24, n))
    day = rng.integers(0, 2, n)
    df["Time"] = (day * 86400 + hours * 3600 + rng.integers(0, 3600, n)).astype(float)
    df["Amount"] = np.where(y == 0, rng.lognormal(3.4, 1.4, n), rng.lognormal(3.0, 1.9, n)).round(2)
    return df.sort_values("Time").reset_index(drop=True)[["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]]

CANDIDATES = ["creditcard.csv", "data/creditcard.csv", "../data/creditcard.csv"]
path = next((p for p in CANDIDATES if os.path.exists(p)), None)

# Not found locally -> try to download the real Kaggle dataset automatically
# (needs internet; if Kaggle asks for credentials, see https://www.kaggle.com/docs/api)
if path is None:
    try:
        import kagglehub                      # pip install kagglehub
        folder = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
        path = os.path.join(folder, "creditcard.csv")
        print("Downloaded dataset via kagglehub ->", path)
    except Exception as e:
        print("Auto-download not available:", type(e).__name__)

if path and os.path.exists(path):
    df = pd.read_csv(path)
    USING_SYNTHETIC = False
    print(f"Loaded real dataset from {path}")
else:
    df = make_synthetic_fraud_data()
    USING_SYNTHETIC = True
    print("WARNING: creditcard.csv not found -> using SYNTHETIC data (same schema). "
          "Download the real file from Kaggle for the genuine benchmark.")

print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()), "| Duplicated rows:", int(df.duplicated().sum()))
df.head()

In [ ]:
counts = df["Class"].value_counts().sort_index()
pct = df["Class"].value_counts(normalize=True).sort_index() * 100
print(f"Legitimate : {counts[0]:>9,}  ({pct[0]:.4f}%)")
print(f"Fraudulent : {counts[1]:>9,}  ({pct[1]:.4f}%)")
print(f"Imbalance ratio: 1 fraud for every {counts[0] / counts[1]:,.0f} legitimate transactions")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=["Legitimate", "Fraud"], y=counts.values, ax=ax[0], palette=["#4C72B0", "#C44E52"])
ax[0].set_title("Transaction counts"); ax[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    ax[0].text(i, v, f"{v:,}", ha="center", va="bottom")
sns.barplot(x=["Legitimate", "Fraud"], y=counts.values, ax=ax[1], palette=["#4C72B0", "#C44E52"])
ax[1].set_yscale("log"); ax[1].set_title("Same counts, log scale (fraud barely visible on linear)")
plt.tight_layout(); plt.show()

**Finding:** fraud is a tiny sliver of all transactions (about **0.17%** in the real Kaggle data: 492 of 284,807). A model can ignore the minority class and still look excellent on naive metrics. That is the core challenge here.

## 2. Why accuracy is a misleading metric on imbalanced data

Accuracy = (TP + TN) / all predictions. With about 99.8% of transactions legitimate, a "model" that **always predicts "not fraud"** scores about **99.8% accuracy** while catching **zero** fraud. It is useless in practice, yet looks near-perfect.

The deeper issue is that accuracy weights every error equally, but the two error types have very different costs:

| Error | Meaning | Typical cost |
|---|---|---|
| **False Negative** | fraud slips through | full transaction value lost, chargeback fees, customer trust |
| **False Positive** | legit transaction blocked | customer friction, a support call, possibly a lost customer |

Because the majority class dominates the denominator, even a large *relative* failure on fraud barely moves accuracy. For example, missing 50% of all fraud costs only ~0.08 percentage points of accuracy. We therefore evaluate with **Precision, Recall, F1, ROC-AUC and PR-AUC**, which focus on the minority class. Let's demonstrate this concretely:

In [ ]:
X_all, y_all = df.drop(columns="Class"), df["Class"]
dummy = DummyClassifier(strategy="most_frequent").fit(X_all, y_all)
pred = dummy.predict(X_all)
print("'Always predict legitimate' baseline")
print(f"  Accuracy : {accuracy_score(y_all, pred):.4%}   <-- looks fantastic")
print(f"  Recall   : {recall_score(y_all, pred):.4%}   <-- catches NO fraud")
print(f"  Precision: {precision_score(y_all, pred, zero_division=0):.4%}")
print(f"  F1       : {f1_score(y_all, pred):.4f}")

## 3. Exploratory data analysis

### 3.1 Transaction amount: fraud vs. non-fraud

In [ ]:
print(df.groupby("Class")["Amount"].describe().round(2))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

bins = np.logspace(-2, np.log10(df["Amount"].max() + 1), 50)
for cls, color, label in [(0, "#4C72B0", "Legitimate"), (1, "#C44E52", "Fraud")]:
    ax[0].hist(df.loc[df.Class == cls, "Amount"].clip(lower=0.01), bins=bins, alpha=0.6,
               color=color, label=label, density=True)
ax[0].set_xscale("log"); ax[0].set_xlabel("Amount (log scale)"); ax[0].set_ylabel("Density")
ax[0].set_title("Amount distribution (normalised per class)"); ax[0].legend()

sns.boxplot(data=df, x="Class", y="Amount", ax=ax[1], showfliers=False, palette=["#4C72B0", "#C44E52"])
ax[1].set_xticklabels(["Legitimate", "Fraud"]); ax[1].set_title("Amount boxplot (outliers hidden)")

sns.violinplot(data=df.assign(logAmount=np.log1p(df.Amount)), x="Class", y="logAmount", ax=ax[2],
               palette=["#4C72B0", "#C44E52"], cut=0)
ax[2].set_xticklabels(["Legitimate", "Fraud"]); ax[2].set_title("log(1 + Amount) by class")
plt.tight_layout(); plt.show()

**Observations (real dataset):** fraudulent transactions are mostly **small to moderate** (median is lower than for legitimate transactions), consistent with "card testing", where criminals probe a stolen card with small charges before larger ones. There is a long right tail in both classes, so Amount is heavily skewed. We will use a **RobustScaler** (median/IQR-based), which is not distorted by those large outliers. Amount alone does **not** separate the classes well; the PCA components carry most of the signal.

### 3.2 Time-of-day analysis

In [ ]:
# 'Time' = seconds elapsed since the first transaction in the dataset (spans ~2 days).
# We fold it into hour-of-day (relative to the start of the capture window).
df["Hour"] = ((df["Time"] // 3600) % 24).astype(int)

hourly = df.groupby("Hour")["Class"].agg(total="count", fraud="sum")
hourly["fraud_rate_%"] = hourly["fraud"] / hourly["total"] * 100

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
ax[0].bar(hourly.index, hourly["total"], color="#4C72B0"); ax[0].set_title("All transactions per hour")
ax[0].set_xlabel("Hour of day"); ax[0].set_ylabel("Count")

ax[1].bar(hourly.index, hourly["fraud"], color="#C44E52"); ax[1].set_title("Fraud transactions per hour")
ax[1].set_xlabel("Hour of day"); ax[1].set_ylabel("Fraud count")

ax[2].plot(hourly.index, hourly["fraud_rate_%"], marker="o", color="#8172B2")
ax[2].axhline(df["Class"].mean() * 100, ls="--", color="gray", label="overall rate")
ax[2].set_title("Fraud rate per hour (%)"); ax[2].set_xlabel("Hour of day"); ax[2].legend()
plt.tight_layout(); plt.show()

hourly.sort_values("fraud_rate_%", ascending=False).head(5).round(3)

**Observations (real dataset):** legitimate volume follows a clear day/night cycle, dropping sharply overnight, while fraud is spread much more evenly across the 24 hours. Because normal traffic is low at night, the **fraud *rate* spikes in the small hours** even though absolute fraud counts are modest. Hour-of-day is therefore a useful engineered feature (`Hour`). *Caveat:* the dataset doesn't state the clock time of the first transaction, so "hour" is relative to the capture start and shouldn't be over-interpreted as true local time. Rates in low-volume hours are also noisy (few fraud cases per hour bucket).

## 4. Preprocessing and stratified train/test split

* `Amount` is skewed -> **RobustScaler**. `Time` (absolute seconds) is dropped in favour of the cyclical-friendly `Hour`.
* `V1`-`V28` are already PCA outputs (zero-mean, comparable scale), so they are left as is.
* **Stratified split** (`stratify=y`) guarantees the same fraud ratio in train and test. Without it, a random 20% split of ~492 fraud cases could by chance contain very few of them, making evaluation unreliable.
* **Order matters:** we split *first*, then fit the scaler on train only and resample train only. Applying SMOTE before the split would leak synthetic copies of test-set neighbours into training and inflate every metric.

In [ ]:
features = [c for c in df.columns if c not in ("Class", "Time")]
X, y = df[features].copy(), df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

# Fit scaler on TRAIN only (avoid leakage), apply to both
scaler = RobustScaler().fit(X_train[["Amount"]])
X_train["Amount"] = scaler.transform(X_train[["Amount"]])
X_test["Amount"] = scaler.transform(X_test[["Amount"]])

summary = pd.DataFrame({
    "rows": [len(y_train), len(y_test)],
    "fraud": [y_train.sum(), y_test.sum()],
    "fraud_%": [y_train.mean() * 100, y_test.mean() * 100]}, index=["train", "test"]).round(4)
print(summary)
assert y_train.sum() > 0 and y_test.sum() > 0, "Fraud must appear in both splits"

## 5. Handling class imbalance

We compare two families of techniques (both listed in the brief):

1. **`class_weight='balanced'`**: no data change; the loss penalises minority-class errors more heavily (weight is inversely proportional to class frequency).
2. **SMOTE** (Synthetic Minority Over-sampling Technique): creates *synthetic* fraud examples by interpolating between a fraud sample and its k nearest fraud neighbours. Fully balancing 1 : 580 would create ~227k synthetic points from ~394 real ones, which is both slow and risks over-fitting to the synthetic manifold. So we use the commonly recommended **hybrid**: SMOTE up to a 1 : 10 minority ratio, then random under-sampling of the majority down to 1 : 2.

Resampling is applied **to the training set only**. The test set keeps its natural, real-world imbalance, so the metrics reflect production conditions.

In [ ]:
resampler = ImbPipeline([
    ("smote", SMOTE(sampling_strategy=0.10, k_neighbors=5, random_state=RANDOM_STATE)),
    ("under", RandomUnderSampler(sampling_strategy=0.50, random_state=RANDOM_STATE)),
])
X_res, y_res = resampler.fit_resample(X_train, y_train)

before = y_train.value_counts().sort_index(); after = pd.Series(y_res).value_counts().sort_index()
print(pd.DataFrame({"original train": before, "after SMOTE + undersample": after}).rename(
    index={0: "legit (0)", 1: "fraud (1)"}))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a, data, t in [(ax[0], before, "Original training set"), (ax[1], after, "After SMOTE + undersampling")]:
    sns.barplot(x=["Legit", "Fraud"], y=data.values, ax=a, palette=["#4C72B0", "#C44E52"]); a.set_title(t)
    for i, v in enumerate(data.values): a.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout(); plt.show()

## 6. Model training

Two model families, each trained under both imbalance strategies (4 models in total):

| Model | Imbalance strategy |
|---|---|
| Logistic Regression | `class_weight='balanced'` |
| Logistic Regression | SMOTE + undersampled data |
| Random Forest | `class_weight='balanced_subsample'` |
| Random Forest | SMOTE + undersampled data |

Logistic Regression is a fast, interpretable linear baseline; Random Forest captures non-linear interactions among the PCA components.

In [ ]:
models, fit_times = {}, {}

def fit(name, model, Xf, yf):
    t0 = time.time(); model.fit(Xf, yf); fit_times[name] = time.time() - t0; models[name] = model
    print(f"{name:<28} trained in {fit_times[name]:6.1f}s")

fit("LogReg + class_weight", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE),
    X_train, y_train)
fit("LogReg + SMOTE", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), X_res, y_res)
fit("RandForest + class_weight", RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=2,
    class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE), X_train, y_train)
fit("RandForest + SMOTE", RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=2,
    n_jobs=-1, random_state=RANDOM_STATE), X_res, y_res)

## 7. Evaluation on the untouched test set

Metrics reported: **Precision, Recall, F1** (at the default 0.5 threshold), plus threshold-independent **ROC-AUC** and **PR-AUC** (average precision). On heavily imbalanced data, PR-AUC is usually the more informative of the two: ROC-AUC can look excellent because the huge number of true negatives keeps the false-positive *rate* tiny even when precision is poor.

In [ ]:
rows, probas = [], {}
for name, m in models.items():
    p = m.predict_proba(X_test)[:, 1]; probas[name] = p
    pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append(dict(Model=name, Accuracy=accuracy_score(y_test, pred),
                     Precision=precision_score(y_test, pred, zero_division=0),
                     Recall=recall_score(y_test, pred), F1=f1_score(y_test, pred),
                     ROC_AUC=roc_auc_score(y_test, p), PR_AUC=average_precision_score(y_test, p),
                     TP=tp, FP=fp, FN=fn, TN=tn))
results = pd.DataFrame(rows).set_index("Model").round(4)
results.style.background_gradient(cmap="Greens", subset=["Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
for name in models:
    fpr, tpr, _ = roc_curve(y_test, probas[name])
    ax[0].plot(fpr, tpr, label=f"{name} (AUC={results.loc[name, 'ROC_AUC']:.3f})")
    pr, rc, _ = precision_recall_curve(y_test, probas[name])
    ax[1].plot(rc, pr, label=f"{name} (AP={results.loc[name, 'PR_AUC']:.3f})")
ax[0].plot([0, 1], [0, 1], "k--", label="random"); ax[0].set_xlabel("False Positive Rate"); ax[0].set_ylabel("True Positive Rate (Recall)")
ax[0].set_title("ROC curves"); ax[0].legend(loc="lower right")
ax[1].axhline(y_test.mean(), color="k", ls="--", label=f"no-skill ({y_test.mean():.4f})")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Precision-Recall curves"); ax[1].legend(loc="upper right")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
for a, (name, p) in zip(axes, probas.items()):
    cm = confusion_matrix(y_test, (p >= 0.5).astype(int))
    sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=a,
                xticklabels=["Pred legit", "Pred fraud"], yticklabels=["Actual legit", "Actual fraud"])
    a.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()

best = results["PR_AUC"].idxmax()
print(f"Best model by PR-AUC: {best}\n")
print(classification_report(y_test, (probas[best] >= 0.5).astype(int), target_names=["Legit", "Fraud"], digits=4))

**Reading the results.** Compare the models on the minority-class metrics, not accuracy (every row has accuracy > 99%). Typical patterns on the real dataset:

* `class_weight='balanced'` and heavy oversampling push **recall up but precision down**: the model flags many legitimate transactions to catch more fraud.
* Random Forest generally dominates Logistic Regression on **PR-AUC and F1** because fraud patterns in the PCA space are non-linear.
* ROC-AUC values look close to each other (~0.97-0.98+), while PR-AUC separates the models much more clearly. This is why ROC-AUC alone is insufficient here.

## 8. Which metric matters most? Recall vs. precision

* **Recall** = TP / (TP + FN): *of all real fraud, how much did we catch?*
* **Precision** = TP / (TP + FP): *of everything we flagged, how much was really fraud?*

**Recall is generally the primary metric for fraud detection**, because a missed fraud (false negative) is usually far more expensive than a false alarm: the bank typically absorbs the full loss plus chargeback and investigation costs, while a false positive costs a declined-card inconvenience or a verification SMS.

**But recall cannot be optimised in isolation.** Flagging *every* transaction gives 100% recall and ~0.17% precision, which is operationally useless: analysts drown in alerts and genuine customers are blocked. So the real objective is *"maximise recall subject to an acceptable precision / alert-volume budget."* This is why we also track:

* **F1 / F2**: F1 balances both; **F2** weights recall twice as much as precision, which suits fraud.
* **PR-AUC**: summarises the entire precision-recall trade-off across thresholds, and is robust to imbalance.
* **Business cost**: ultimately `cost = FN x avg_fraud_loss + FP x review_cost`. The decision threshold should minimise this, not default to 0.5.

The next cell makes the trade-off concrete by moving the decision threshold.

In [ ]:
best_p = probas[best]
prec, rec, thr = precision_recall_curve(y_test, best_p)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(thr, prec[:-1], label="Precision"); ax[0].plot(thr, rec[:-1], label="Recall")
f1s = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-9, None)
ax[0].plot(thr, f1s, label="F1", ls="--")
ax[0].set_xlabel("Decision threshold"); ax[0].set_title(f"Precision / Recall / F1 vs. threshold ({best})"); ax[0].legend()

# Illustrative cost model (assumptions - tune to your business!)
AVG_FRAUD_LOSS, REVIEW_COST = 120.0, 5.0
thresholds = np.linspace(0.02, 0.98, 97)
costs = []
for t in thresholds:
    tn, fp, fn, tp = confusion_matrix(y_test, (best_p >= t).astype(int), labels=[0, 1]).ravel()
    costs.append(fn * AVG_FRAUD_LOSS + (fp + tp) * REVIEW_COST)
t_opt = thresholds[int(np.argmin(costs))]
ax[1].plot(thresholds, costs); ax[1].axvline(t_opt, color="r", ls="--", label=f"min cost @ t={t_opt:.2f}")
ax[1].set_xlabel("Decision threshold"); ax[1].set_ylabel("Total cost"); ax[1].set_title(
    f"Illustrative cost curve (fraud loss=${AVG_FRAUD_LOSS:.0f}, review=${REVIEW_COST:.0f})"); ax[1].legend()
plt.tight_layout(); plt.show()

tbl = []
for t in [0.1, 0.2, 0.3, 0.5, 0.7, t_opt]:
    pr_ = (best_p >= t).astype(int)
    tbl.append(dict(threshold=round(t, 2), precision=precision_score(y_test, pr_, zero_division=0),
                    recall=recall_score(y_test, pr_), F1=f1_score(y_test, pr_),
                    alerts=int(pr_.sum()), missed_fraud=int(((pr_ == 0) & (y_test == 1)).sum())))
pd.DataFrame(tbl).round(3)

## 9. Feature importance and coefficient analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 7))

lr = models["LogReg + class_weight"]
coef = pd.Series(lr.coef_[0], index=X_train.columns).sort_values()
top = pd.concat([coef.head(10), coef.tail(10)])
ax[0].barh(top.index, top.values, color=["#C44E52" if v > 0 else "#4C72B0" for v in top.values])
ax[0].set_title("Logistic Regression: top 20 coefficients\n(red = pushes toward fraud, blue = toward legit)")
ax[0].set_xlabel("Coefficient (features are on comparable scales)")

rf = models["RandForest + SMOTE"]
imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)[::-1]
ax[1].barh(imp.index, imp.values, color="#55A868"); ax[1].set_title("Random Forest: top 15 impurity-based importances")
ax[1].set_xlabel("Importance")
plt.tight_layout(); plt.show()

print("Top 5 RF features:", list(imp.index[::-1][:5]))

In [ ]:
# Impurity-based importance is biased toward high-cardinality / continuous features.
# Permutation importance on the test set (scored by PR-AUC) is a more trustworthy cross-check.
from sklearn.inspection import permutation_importance
sample_idx = np.random.RandomState(0).choice(len(X_test), size=min(len(X_test), 20_000), replace=False)
# keep all fraud rows in the sample so the score is meaningful
fraud_idx = np.where(y_test.values == 1)[0]
idx = np.union1d(sample_idx, fraud_idx)
pi = permutation_importance(rf, X_test.iloc[idx], y_test.iloc[idx], scoring="average_precision",
                            n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
pi_s = pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False).head(10)[::-1]
pi_s.plot.barh(color="#8172B2", figsize=(7, 4), title="Permutation importance (drop in PR-AUC), Random Forest")
plt.tight_layout(); plt.show()

**Interpretation notes**

* Because `V1`-`V28` are anonymised PCA components, we can say *which* components matter (on the real dataset typically `V14`, `V10`, `V12`, `V17`, `V4`, `V11`) but not their business meaning. This is a privacy trade-off of the benchmark; with raw features (merchant category, country, device, velocity counts) the analysis would be directly actionable.
* **Logistic Regression coefficients** give direction and magnitude (sign = increases/decreases fraud log-odds), which makes them easy to explain to regulators and analysts.
* **Random Forest importances** show non-linear usefulness but not direction. Permutation importance confirms which features the model truly relies on for *ranking* fraud.

## 10. Scalability: handling 1 million transactions per hour

**Sizing.** 1,000,000 / hour = **~278 transactions per second (TPS)** on average. Real traffic is bursty (lunch hours, holidays, Black Friday), so plan for **5-10x peak, i.e. ~1,500-3,000 TPS**. The key requirement for card payments is a *real-time* decision, typically **< 50-100 ms** per transaction, since scoring happens inside the authorisation flow.

Let's measure the raw inference throughput of our models on this machine:

In [ ]:
needed_avg_tps = 1_000_000 / 3600
print(f"Required average throughput: {needed_avg_tps:,.0f} tx/s   (peak planning ~10x = {needed_avg_tps * 10:,.0f} tx/s)\n")

bench = []
for name, m in models.items():
    # (a) batch scoring of the whole test set
    t0 = time.perf_counter(); m.predict_proba(X_test); batch_s = time.perf_counter() - t0
    # (b) single-transaction online scoring latency (worst case: no micro-batching)
    singles = []
    for i in range(200):
        row = X_test.iloc[[i]]
        t0 = time.perf_counter(); m.predict_proba(row); singles.append((time.perf_counter() - t0) * 1000)
    bench.append(dict(Model=name, batch_tx_per_sec=len(X_test) / batch_s,
                      single_p50_ms=np.percentile(singles, 50), single_p99_ms=np.percentile(singles, 99)))
bench = pd.DataFrame(bench).set_index("Model").round(2)
bench["x_required_batch"] = (bench["batch_tx_per_sec"] / needed_avg_tps).round(1)
bench

**What the benchmark tells us.** Batch inference on a single machine is comfortably above 278 TPS for both models (the linear model by orders of magnitude). The single-row latency is dominated by framework overhead (scikit-learn is not designed for per-row calls); a production service would use **micro-batching** or an exported/compiled model. So the *model itself* is not the bottleneck at 1M tx/hour; the surrounding system is.

### Production architecture for 1M tx/hour

| Concern | Approach |
|---|---|
| **Ingestion** | Stream transactions through **Kafka / Kinesis**, partitioned by card or account ID so all events for one card go to the same consumer (preserves ordering and enables stateful features). |
| **Feature computation** | Real-time **velocity/aggregate features** (tx count in the last 1 min / 1 h / 24 h, distance from last transaction, new merchant/country flags) via **Flink / Spark Structured Streaming**, backed by a low-latency **feature store** (Redis / Feast) to keep training and serving features consistent. |
| **Model serving** | Stateless scoring microservice (FastAPI / **ONNX Runtime** / Triton / Treelite-compiled trees), **horizontally scaled** behind a load balancer with autoscaling on queue depth. A single 4-8 core instance handles thousands of TPS for tree models; run N replicas for redundancy and peaks. |
| **Latency control** | Micro-batching (e.g. 5-10 ms windows), model compression (fewer/shallower trees), **cascade design**: a cheap Logistic Regression / rules layer clears the ~95% of obviously safe traffic, and only suspicious transactions go to the heavier model. Hard timeout with a safe fallback decision. |
| **Resampling** | **SMOTE is a training-time technique only**: it adds no cost at inference. Training can be done offline on a sampled/under-sampled dataset (SMOTE's nearest-neighbour search scales poorly on millions of rows, so under-sample the majority first or train on a rolling window). |
| **Training at scale** | Scheduled retraining on Spark / distributed XGBoost / LightGBM; incremental learners (`partial_fit`, online gradient boosting) for fast adaptation. |
| **Concept drift** | Fraudsters adapt, so performance decays. Monitor input-feature drift (PSI), score distribution, and recall/precision on **delayed labels** (chargebacks arrive days or weeks later). Retrain on a schedule or on drift alert; use champion/challenger or shadow deployment before promotion. |
| **Human-in-the-loop** | Route mid-confidence scores to analyst queues or step-up authentication (OTP) rather than hard-declining, which controls the false-positive burden and generates new labels. |
| **Reliability & compliance** | Multi-AZ deployment, idempotent consumers, audit logging, explainability (SHAP / reason codes) for disputes and regulation, PII protection. |

**Bottom line:** at ~278 TPS average (a few thousand at peak), a tree-ensemble or linear model is easily fast enough. The engineering challenges are *feature freshness, consistent online/offline features, latency SLAs, drift, and label delay*, not raw model throughput.

## 11. Summary and conclusions

1. **Imbalance:** fraud is ~0.17% of transactions (492 / 284,807 in the real data): a 1 : 580 ratio.
2. **Accuracy is meaningless here:** an "always legit" classifier scores ~99.8% while detecting no fraud. We use Precision, Recall, F1, ROC-AUC and (especially) PR-AUC.
3. **EDA:** fraud skews toward smaller amounts and is more evenly spread through the day, so its *rate* peaks in low-traffic night hours.
4. **Methodology:** stratified split; scaler and resampling fitted on the **training set only** to avoid leakage; test set left at its natural imbalance.
5. **Imbalance techniques:** `class_weight` and SMOTE + undersampling both lift recall relative to an unweighted model; the trade-off is more false positives.
6. **Models:** Random Forest typically beats Logistic Regression on PR-AUC and F1; Logistic Regression remains valuable as an interpretable, ultra-fast baseline or first-stage filter.
7. **Metric priority:** maximise **recall** at an acceptable **precision**, with the threshold chosen by business cost rather than the default 0.5.
8. **Scalability:** 1M tx/hour is only ~278 TPS; the model is not the bottleneck. Streaming features, low-latency serving, drift monitoring and delayed-label retraining are what matter.

**Possible next steps:** XGBoost / LightGBM with `scale_pos_weight`, cost-sensitive learning, probability calibration, anomaly-detection (Isolation Forest / autoencoder) as a complementary unsupervised signal, nested stratified cross-validation for tighter confidence intervals on a test set with only ~98 fraud cases, and SHAP for per-transaction explanations.

**References:** imbalanced-learn docs ([imbalanced-learn.org](https://imbalanced-learn.org)); Chawla et al. (2002), *SMOTE: Synthetic Minority Over-sampling Technique*; Saito & Rehmsmeier (2015), *The Precision-Recall Plot Is More Informative than the ROC Plot on Imbalanced Datasets*.